In [0]:
/*
Static lookup tables to represent fiscal quarters.
*/
with dates as (
  select cast(m as date) as m, fq, fy, q
  , last_day(m) as last_day_of_month, year(m) as cy, month(m) as cm
  from (
    values 
      /*('2025-02-01', 'FY26-Q1', 2026, 1),
      ('2025-03-01', 'FY26-Q1', 2026, 1),
      ('2025-04-01', 'FY26-Q1', 2026, 1),
      ('2025-05-01', 'FY26-Q2', 2026, 2),
      ('2025-06-01', 'FY26-Q2', 2026, 2),
      ('2025-07-01', 'FY26-Q2', 2026, 2),
      ('2025-08-01', 'FY26-Q3', 2026, 3),
      ('2025-09-01', 'FY26-Q3', 2026, 3),
      ('2025-10-01', 'FY26-Q3', 2026, 3),
      ('2025-11-01', 'FY26-Q4', 2026, 4),
      ('2025-12-01', 'FY26-Q4', 2026, 4),
      ('2026-01-01', 'FY26-Q4', 2026, 4),*/
      ('2026-02-01', 'FY27-Q1', 2027, 1),
      ('2026-03-01', 'FY27-Q1', 2027, 1),
      ('2026-04-01', 'FY27-Q1', 2027, 1),
      ('2026-05-01', 'FY27-Q2', 2027, 2),
      ('2026-06-01', 'FY27-Q2', 2027, 2),
      ('2026-07-01', 'FY27-Q2', 2027, 2),
      ('2026-08-01', 'FY27-Q3', 2027, 3),
      ('2026-09-01', 'FY27-Q3', 2027, 3),
      ('2026-10-01', 'FY27-Q3', 2027, 3),
      ('2026-11-01', 'FY27-Q4', 2027, 4),
      ('2026-12-01', 'FY27-Q4', 2027, 4),
      ('2027-01-01', 'FY27-Q4', 2027, 4) 
  ) as dates(m, fq, fy, q)
),

financial_quarters as (
  select fq, fy, q
    , max(last_day_of_month) as fiscal_quarter_end_date
    , min(m) as fiscal_quarter_start_date
    , case when getdate() > fiscal_quarter_end_date then q else null end as last_closed_q
    , case when getdate() > fiscal_quarter_end_date then true else false end as is_quarter_closed
    , sum(day(last_day_of_month)) as days_in_quarter
    , case when getdate() >= fiscal_quarter_start_date and current_date() <= fiscal_quarter_end_date then true else false end as is_current_fiscal_quarter
    , case when current_date() >= make_date(fy - 1, 2, 1) and current_date() <= make_date(fy, 1, 31) then 1 else 0 end as is_current_fiscal_year
    , (select max(usage_date) from main.gtm_gold.individual_consumption_daily)  as latest_usage_date
    , greatest(0, least(days_in_quarter, datediff(fiscal_quarter_end_date, latest_usage_date))) as days_left_in_quarter
    , case when is_current_fiscal_quarter then q else 0 end as current_quarter_number
  from dates
  group by fq, fy, q
),

blockers AS (
  SELECT
    b.usecase_id,
    SUM(case when b.type = 'Blocked' or isnull(b.type) then 1 else 0 end) as blocked_count,
    SUM(case when b.type = 'Friction' then 1 else 0 end) as friction_count,
    ARRAY_JOIN(
      ARRAY_DISTINCT(FILTER(COLLECT_LIST(
        ARRAY_JOIN(
          FILTER(
            ARRAY(
              COALESCE('<a href="https://databrickinternal.ideas.aha.io/ideas/' || aha_item.aha_reference || '" target="_blank">' || aha_item.aha_reference || '</a>', ''),
              COALESCE(aha_item.aha_name, ''),
              COALESCE(b.comment, '')
            ),
            x -> x != ''
          ),
          ' - '
        )
      ), x -> x IS NOT NULL AND x != '')),
      '. '
    ) AS blocker_details
  FROM main.gtm_silver.blocker_detail b
  LATERAL VIEW OUTER EXPLODE(b.aha) AS aha_item
  WHERE b.concatenated_emails like '%' || :ae_email || '%'
  --AND b.snapshot_date = current_date()
  GROUP BY b.usecase_id
),

 use_case_pipeline_changes_filtered as (
  select
    pop.usecase_id,
    pop.stage_number_latest,
    pop.stage_number_prior,
    pop.target_live_date_latest,
    pop.target_live_date_prior,
    pop.target_live_fiscal_year_quarter,
    pop.estimated_quarterly_dollar_dbus_latest,
    pop.estimated_quarterly_dollar_dbus_prior,
    case when pop.stage_number_prior is null then 1
           when pop.stage_number_latest > pop.stage_number_prior then 1
           when pop.stage_number_latest = pop.stage_number_prior then 0
           else -1 end as stage_advanced,
    case when pop.stage_number_prior is null then 'New in pipeline'
           when pop.stage_number_latest > pop.stage_number_prior then concat('Advanced from stage ', cast(pop.stage_number_prior as string))
           when pop.stage_number_latest = pop.stage_number_prior then 'No stage change'
           else concat('Regressed from stage ', cast(pop.stage_number_prior as string))
      end as stage_change_description,
    case when pop.target_live_date_prior is null then 1
           when pop.target_live_date_latest < pop.target_live_date_prior then 1
           when pop.target_live_date_latest = pop.target_live_date_prior then 0
           else -1 end as target_date_pulled_foreward,
    case when pop.target_live_date_prior is null then 'New in pipeline'
           when pop.target_live_date_latest < pop.target_live_date_prior then concat('Pulled forward by ', cast(abs(datediff(pop.target_live_date_latest, pop.target_live_date_prior)) as string), ' days from ', cast(pop.target_live_date_prior as string))
           when pop.target_live_date_latest = pop.target_live_date_prior then 'No target date change'
           else concat('Pushed back by ', cast(datediff(pop.target_live_date_latest, pop.target_live_date_prior) as string), ' days from ', cast(pop.target_live_date_prior as string))
      end as target_date_change_description,
    datediff(pop.target_live_date_latest, pop.target_live_date_prior) as target_live_date_diff_days,
    case when pop.estimated_quarterly_dollar_dbus_prior is null then 1
           when pop.estimated_quarterly_dollar_dbus_latest > pop.estimated_quarterly_dollar_dbus_prior then 1
           when pop.estimated_quarterly_dollar_dbus_latest = pop.estimated_quarterly_dollar_dbus_prior then 0
           else -1 end as amount_increased,
    case when pop.estimated_quarterly_dollar_dbus_prior is null then 'New in pipeline'
           when pop.estimated_quarterly_dollar_dbus_latest > pop.estimated_quarterly_dollar_dbus_prior then concat('Increased by ', chr(36), cast(cast(pop.estimated_quarterly_dollar_dbus_latest - pop.estimated_quarterly_dollar_dbus_prior as bigint) as string), ' per quarter')
           when pop.estimated_quarterly_dollar_dbus_latest = pop.estimated_quarterly_dollar_dbus_prior then 'No amount change'
           else concat('Decreased by ', chr(36), cast(cast(abs(pop.estimated_quarterly_dollar_dbus_latest - pop.estimated_quarterly_dollar_dbus_prior) as bigint) as string), ' per quarter')
      end as amount_change_description,
    pop.estimated_quarterly_dollar_dbus_latest - pop.estimated_quarterly_dollar_dbus_prior as change_amount
   , case 
      when amount_increased < 0 or target_date_pulled_foreward < 0 or stage_advanced < 0 then 'Negative change' 
      when amount_increased > 0 or target_date_pulled_foreward > 0 or stage_advanced > 0 then 'Positive change'      
   else 'No change' end change_type_label
  from main.gtm_gold.use_case_pipeline_changes as pop
  inner join gtm_silver.use_case_detail as d
    on pop.usecase_id = d.usecase_id
    and pop.estimated_quarterly_dollar_dbus_latest = d.estimated_quarterly_dollar_dbus
    and pop.stage_number_latest = d.stage_number
    and pop.target_live_fiscal_year_quarter = d.target_live_fiscal_year_quarter  
  where pop.period = :period
    and pop.change_type <> 'no change'
    and d.Business_Unit = :business_unit
    and d.sales_subregion_level_1 = :region_level_1
    and d.sales_subregion_level_2 = :region_level_2
),

usecases_filtered as (
  select d.usecase_id, d.usecase_name, d.estimated_monthly_dollar_dbus, d.target_onboarding_date, d.target_live_date
    , nullif(trim(regexp_extract(d.demand_plan_next_steps, '(?s)\\[Risk Mitigation\\](.*?)(\\r?\\n\\s*\\r?\\n|$)', 1)), '') as mitigation_plan
    , nullif(trim(regexp_extract(d.demand_plan_next_steps, '(?s)\\[Acceleration\\](.*?)(\\r?\\n\\s*\\r?\\n|$)', 1)), '') as acceleration_plan
    , coalesce(mitigation_plan, acceleration_plan) as manager_notes
    , regexp_like(d.demand_plan_next_steps, '#keytechwin') as is_keytechwin
    , dateadd(day, 14, date_trunc('month',d.target_onboarding_date)) as target_onboarding_date_15 
    , dateadd(day, 14, date_trunc('month',d.target_live_date)) as target_live_date_15
    , datediff(d.target_live_date, d.target_onboarding_date) as total_ramping_days 
    , date_format(dateadd(year, +1, dateadd(month, -1, d.target_onboarding_date)), "'FY'yy'-Q'Q") as target_onboarding_date_fq
    , date_format(dateadd(year, +1, dateadd(month, -1, d.target_live_date)), "'FY'yy'-Q'Q") as target_live_date_fq
    , concat('<a href="https://databricks.lightning.force.com/lightning/r/UseCase__c/', d.usecase_id, '/view" targe="_blank">', d.usecase_name, '</a>') as usecase_url
    , coalesce(num_of_blockers, 0) as num_of_blockers
    , coalesce(blk.blocked_count, 0) as blocked_count
    , coalesce(blk.friction_count, 0) as friction_count
    , blk.blocker_details
    , coalesce(implementation_status, 'Unknown') as impl_status
    , coalesce(pop.stage_advanced, 0) as stage_advanced
    , coalesce(pop.stage_change_description, 'No stage change') as stage_change_description
    , coalesce(pop.target_date_pulled_foreward, 0) as target_date_pulled_foreward
    , coalesce(pop.target_date_change_description, 'No target date change') as target_date_change_description
    , pop.target_live_date_diff_days
    , coalesce(pop.amount_increased, 0) as amount_increased
    , coalesce(pop.amount_change_description, 'No amount change') as amount_change_description
    , coalesce(pop.change_amount, 0) as change_amount
    , coalesce(pop.change_type_label, 'No change') as change_type_label
    , concat('<a href="', get(FILTER(d.usecase_documents, doc -> doc.document_type = 'Eval Doc'), 0).document_link, '" target="_blank">Eval Doc</a>') as eval_doc_link
    , concat('<a href="', get(FILTER(d.usecase_documents, doc -> doc.document_type = 'Onboarding Doc'), 0).document_link, '" target="_blank">Onboarding Doc</a>') as onboarding_doc_link
    , case
        when d.days_in_stage <= 30 or d.days_in_stage is null then '0-30 days'
        when d.days_in_stage > 30 and d.days_in_stage <= 60 then '31-60 days'
        when d.days_in_stage > 60 and d.days_in_stage <= 120 then '61-120 days'
        when d.days_in_stage > 120 then '120+ days'
      end as days_in_stage_bucket
    , date_diff(DAY, current_date(), last_day(d.target_live_date)) as days_to_go_live
    , case when days_to_go_live < 0 then true else false end go_live_in_the_past
    , date_diff(DAY, current_date(), last_day(d.target_onboarding_date)) as days_to_onboarding

    --Collect all warnings
    , flatten(array(                
        case when days_to_go_live between 0 and 30 and d.stage_number < 5 then array('Go live < 30 / Not U5') else array() end,
        case when days_to_go_live between 0 and 30 and d.implementation_status = 'Red' then array('Go live < 30 days / Red') else array() end,
        case when days_to_go_live between 0 and 30 and d.implementation_status = 'Yellow' then array('Go live < 30 days / Yellow') else array() end,
        case when days_to_go_live between 0 and 30 then array('Go live < 30') else array() end,
        
        case when days_to_onboarding between 0 and 30 and d.stage_number < 5 and d.implementation_status = 'Red' then array('Onboarding < 30 / Red') else array() end,
        case when days_to_onboarding between 0 and 30 and d.stage_number < 5 and d.implementation_status = 'Yellow' then array('Onboarding < 30 / Yellow') else array() end,
        case when days_to_onboarding between 0 and 30 and d.stage_number <= 3 then array('Onboarding < 30 / <=U3') else array() end,
        case when days_to_onboarding between 0 and 30 and d.stage_number = 4 then array('Onboarding < 30 / U4') else array() end
      )) as warning_rules

    -- Collect all Hygiene checks
    , flatten(array(
        case when d.implementation_status is null then array('Health status not defined') else array() end,
        case when days_to_go_live < 0 then array('Go live date in the past') else array() end,
        case when days_to_onboarding < 0 and d.stage_number < 5 then array('Past Onboarding date / not U5') else array() end,
        case when days_to_go_live between 0 and 30 and d.implementation_status = 'Red' then array('Go live < 30 days / Red') else array() end,
        case when eval_doc_link is null and d.estimated_monthly_dollar_dbus >= 10000 and d.stage_number between 2 and 4 then array('Eval doc required') else array() end,
        case when onboarding_doc_link is null and d.estimated_monthly_dollar_dbus >= 10000 and d.stage_number between 4 and 4 then array('Onboarding doc required') else array() end
      )) as hygiene_rules,
      
  case when array_size(warning_rules) = 0 then false else true end as has_warnings,
  case when array_size(hygiene_rules) = 0 then false else true end as has_hygiene_issues
    
  from gtm_silver.use_case_detail as d
  left outer join blockers as blk
  on blk.usecase_id = d.usecase_id
  
  left join use_case_pipeline_changes_filtered as pop
   on pop.usecase_id = d.usecase_id
   and pop.estimated_quarterly_dollar_dbus_latest = d.estimated_quarterly_dollar_dbus
   and pop.stage_number_latest = d.stage_number
   and pop.target_live_fiscal_year_quarter = d.target_live_fiscal_year_quarter

  where d.Business_Unit = :business_unit
    and d.sales_subregion_level_1 = :region_level_1
    and d.sales_subregion_level_2 = :region_level_2
    and d.is_incremental = true --Excludes upgrades
    and d.stage_number <= 5 --Filter out 'Disqualified', 'Lost' and 'Live' UCOs.
    and coalesce(d.estimated_monthly_dollar_dbus, 0) > 0 -- Excludes zero-valued use cases.
    and d.concatenated_emails like '%' || :ae_email || '%'
),

incremental_projections as (
  select uco.usecase_id, d.fq, d.fy, d.q, d.m, d.cm, d.last_day_of_month
    , uco.target_onboarding_date, uco.target_onboarding_date_fq, uco.target_live_date, uco.target_live_date_fq, uco.target_onboarding_date_15, uco.target_live_date_15
    , uco.total_ramping_days, uco.estimated_monthly_dollar_dbus, uco.impl_status, uco.usecase_url, uco.num_of_blockers 
    , (select max(usage_date) from main.gtm_gold.individual_consumption_daily) as latest_usage_date
    , datediff(latest_usage_date, target_onboarding_date_15) as current_ramping_days 
    -- Calculate this month's baseline for each use case, .i.e. how much are they consuming today? This is used to calculate the actual incremental consumption at the next step.
    -- We assume that the onboarding date and live date occur on day 15 of the month.
    ,case when d.m between uco.target_onboarding_date and uco.target_live_date then 1 else 0 end as is_onboarding
  
    ,case         
        -- if UCO not onboarded yet (i.e. the onboarding date is in the future), then no dbus are generated for the current month.
        when target_onboarding_date_15 > latest_usage_date then 0
        -- if UCO is already live, then it should already realise the expected monthly $DBUs.
        when latest_usage_date > target_live_date_15 then estimated_monthly_dollar_dbus
        -- if UCO is currently onboarding (i.e. the onboarding date is in the past), this is the estimated dbus for the full current month. 
        else round(estimated_monthly_dollar_dbus * try_divide(datediff(latest_usage_date, target_onboarding_date_15), total_ramping_days)) 
      end as current_dbu_baseline 

    --calculate the ramping dbus assuming a linear ramp between the tonboarding date and the go-live: from 0 $dbus to the expected monthly $dbus that will be reached on go-live.
    , case       
        when d.last_day_of_month < uco.target_onboarding_date_15 then 0 --Before the onboarding date
        when d.last_day_of_month > uco.target_live_date_15 then uco.estimated_monthly_dollar_dbus --After the go-live the $dbus remain flat 
        else round(uco.estimated_monthly_dollar_dbus * try_divide(datediff(d.last_day_of_month, uco.target_onboarding_date_15), uco.total_ramping_days)) --Between onboarding and go-live the dbus ramp-up linearly
      end as ramping_dbus

    --remove the realised dbus from the ramp, when the use case is ramping up during the onboarding phase, past months' revenue has already been realised.
    , case 
      when d.last_day_of_month < latest_usage_date then 0 --Past month: if a use case started onboarding in the past, and the month is closed then we are removing the consumption from the pipeline to avoid double counting, because we assume it has already been realised (actual dbus).
      when d.last_day_of_month < uco.target_onboarding_date_15 then 0 --Before the onboarding date
      when d.last_day_of_month > uco.target_live_date_15 then uco.estimated_monthly_dollar_dbus - current_dbu_baseline --After the go-live
      else round(uco.estimated_monthly_dollar_dbus * try_divide(datediff(d.last_day_of_month, uco.target_onboarding_date_15), uco.total_ramping_days)) - current_dbu_baseline --Between onboarding and go-live 
    end as ramping_dbus_from_baseline 

     -- calculates the actual incremental value substracting last month's $dbus from this month's $dbus.
    , case 
        when latest_usage_date > m then ramping_dbus - ramping_dbus_from_baseline
        else 0 --in the future
    end as dbus_generated

  from usecases_filtered as uco
  inner join dates as d --cross join with date table
),

quarterly_projection_by_use_case as (
  select              
    i.usecase_id, i.fy, i.fq, i.q
    , sum(i.ramping_dbus) as quarterly_ramping_dbus    
    , sum(i.dbus_generated) as quarterly_dbus_generated
    , lag(max(i.ramping_dbus)) over (partition by i.usecase_id order by i.fq asc) as last_day_of_prev_quarter_dbus
    from incremental_projections as i    
    group by all
),

monthly_projection as (
  select
    ip.usecase_id, ip.fy, ip.fq, ip.q, ip.m, ip.cm, ip.ramping_dbus, ip.current_dbu_baseline, ip.is_onboarding
    , f.last_closed_q, f.days_left_in_quarter, f.is_quarter_closed, f.is_current_fiscal_quarter
    , qp.last_day_of_prev_quarter_dbus    
    , greatest(qp.last_day_of_prev_quarter_dbus, ip.current_dbu_baseline) as quarter_dbu_baseline

    -- Incremental quarterly $dbus. 
    -- If the quarter has started then we use the current baseline to identify addtional incremental dbus until the end of the quarter
    -- if the quarter has not started yet the baseline is the last day of the previous quarter.    
    ,case 
        when ip.ramping_dbus - greatest(qp.last_day_of_prev_quarter_dbus, ip.current_dbu_baseline) < 0 then 0 -- All past months do not contribute to incremental dbus. 
        else ip.ramping_dbus - greatest(qp.last_day_of_prev_quarter_dbus, ip.current_dbu_baseline)
     end as quarterly_incremental_dbus

    , case when ip.impl_status = 'Green' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end as dbus_in_pipeline_green 
    , case when ip.impl_status = 'Yellow' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end as dbus_in_pipeline_yellow 
    , case when ip.impl_status = 'Red' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end as dbus_in_pipeline_red
    , case when ip.impl_status = 'Unknown' and not f.is_quarter_closed then quarterly_incremental_dbus else 0 end as dbus_in_pipeline_unknown
  
  from incremental_projections as ip
  inner join quarterly_projection_by_use_case as qp
  on ip.usecase_id = qp.usecase_id
  and ip.fq = qp.fq
  inner join financial_quarters as f
  on ip.fq = f.fq
),

-- Note: monthly pivot column aliases are prefixed with m_ to avoid digit-starting identifier issues
-- when referencing them via table alias (e.g. mp.m_2025_02 instead of mp.`2025_02`)
monthly_projection_pivoted as (
  select usecase_id
  , coalesce(m_2025_02, 0) as m_2025_02, coalesce(m_2025_03, 0) as m_2025_03, coalesce(m_2025_04, 0) as m_2025_04, coalesce(m_2025_05, 0) as m_2025_05, coalesce(m_2025_06, 0) as m_2025_06, coalesce(m_2025_07, 0) as m_2025_07
  , coalesce(m_2025_08, 0) as m_2025_08, coalesce(m_2025_09, 0) as m_2025_09, coalesce(m_2025_10, 0) as m_2025_10, coalesce(m_2025_11, 0) as m_2025_11, coalesce(m_2025_12, 0) as m_2025_12, coalesce(m_2026_01, 0) as m_2026_01
  , coalesce(m_2026_02, 0) as m_2026_02, coalesce(m_2026_03, 0) as m_2026_03, coalesce(m_2026_04, 0) as m_2026_04, coalesce(m_2026_05, 0) as m_2026_05, coalesce(m_2026_06, 0) as m_2026_06, coalesce(m_2026_07, 0) as m_2026_07
  , coalesce(m_2026_08, 0) as m_2026_08, coalesce(m_2026_09, 0) as m_2026_09, coalesce(m_2026_10, 0) as m_2026_10, coalesce(m_2026_11, 0) as m_2026_11, coalesce(m_2026_12, 0) as m_2026_12, coalesce(m_2027_01, 0) as m_2027_01
  from 
  (
    select usecase_id, m, ramping_dbus from monthly_projection
  )
  pivot (sum(ramping_dbus) AS dbus
      for m in (
        '2025-02-01' as m_2025_02,
        '2025-03-01' as m_2025_03,
        '2025-04-01' as m_2025_04,
        '2025-05-01' as m_2025_05,
        '2025-06-01' as m_2025_06,
        '2025-07-01' as m_2025_07,
        '2025-08-01' as m_2025_08,
        '2025-09-01' as m_2025_09,
        '2025-10-01' as m_2025_10,
        '2025-11-01' as m_2025_11,
        '2025-12-01' as m_2025_12,
        '2026-01-01' as m_2026_01,
        '2026-02-01' as m_2026_02,
        '2026-03-01' as m_2026_03,
        '2026-04-01' as m_2026_04,
        '2026-05-01' as m_2026_05,
        '2026-06-01' as m_2026_06,
        '2026-07-01' as m_2026_07,
        '2026-08-01' as m_2026_08,
        '2026-09-01' as m_2026_09,
        '2026-10-01' as m_2026_10,
        '2026-11-01' as m_2026_11,
        '2026-12-01' as m_2026_12,
        '2027-01-01' as m_2027_01
      )
  )
),

quartely_projection_pivoted as (
  select usecase_id
  , coalesce(FY26_Q1, 0) FY26_Q1
  , coalesce(FY26_Q2, 0) FY26_Q2
  , coalesce(FY26_Q3, 0) FY26_Q3
  , coalesce(FY26_Q4, 0) FY26_Q4
  , coalesce(FY27_Q1, 0) FY27_Q1
  , coalesce(FY27_Q2, 0) FY27_Q2
  , coalesce(FY27_Q3, 0) FY27_Q3
  , coalesce(FY27_Q4, 0) FY27_Q4
  from 
  (
    select usecase_id, fq, sum(quarterly_incremental_dbus) as quarterly_incremental_dbus 
    from monthly_projection
    group by all
  )
  pivot (sum(quarterly_incremental_dbus) AS dbus
      for fq in (
        'FY26-Q1' as FY26_Q1,
        'FY26-Q2' as FY26_Q2,
        'FY26-Q3' as FY26_Q3,
        'FY26-Q4' as FY26_Q4,
        'FY27-Q1' as FY27_Q1,
        'FY27-Q2' as FY27_Q2,
        'FY27-Q3' as FY27_Q3,
        'FY27-Q4' as FY27_Q4
      )
  )
),

uco_view as (
  select c.sales_subregion_level_1, c.sales_subregion_level_2, c.sales_subregion_level_3, c.account_name, c.deployable_account_name, c.account_executive, c.solution_architect, c.dsa_user_name as dsa, c.arr_band, c.usecase_id, c.usecase_name 
    , c.target_onboarding_date, b.target_onboarding_date_fq, c.target_live_date, b.target_live_date_fq, c.target_cloud, c.is_migration_usecase
    , nullif(trim(c.migration_source_platform), '') as migration_source_platform_adj
    , c.is_incremental, b.is_keytechwin
    , c.stage, c.stage_number, c.stage_name_ui, c.days_in_stage, c.days_in_validating, c.days_in_scoping, c.days_in_evaluating, c.days_in_confirming, c.days_in_onboarding
    , c.usecase_description, c.demand_plan_next_steps, c.implementation_notes
    , c.implementation_partner_name, c.has_ps_project, c.use_case_product_enriched
    , b.estimated_monthly_dollar_dbus, c.estimated_monthly_dollar_dbus_weighted, b.total_ramping_days
    , c.estimated_quarterly_dollar_dbus, c.estimated_quarterly_dollar_dbus_weighted
    , b.days_in_stage_bucket, b.usecase_url, b.eval_doc_link, b.onboarding_doc_link, b.impl_status as implementation_status, b.num_of_blockers, b.blocked_count, b.friction_count, b.blocker_details
    , b.stage_advanced, b.stage_change_description, b.target_date_pulled_foreward, b.target_date_change_description, b.target_live_date_diff_days, b.amount_increased, b.amount_change_description, b.change_amount, b.change_type_label
    , b.manager_notes
    , qp.FY26_Q1, qp.FY26_Q2, qp.FY26_Q3, qp.FY26_Q4, qp.FY27_Q1, qp.FY27_Q2, qp.FY27_Q3, qp.FY27_Q4
    , mp.m_2025_02, mp.m_2025_03, mp.m_2025_04, mp.m_2025_05, mp.m_2025_06, mp.m_2025_07, mp.m_2025_08, mp.m_2025_09, mp.m_2025_10, mp.m_2025_11, mp.m_2025_12
    , mp.m_2026_01, mp.m_2026_02, mp.m_2026_03, mp.m_2026_04, mp.m_2026_05, mp.m_2026_06, mp.m_2026_07, mp.m_2026_08, mp.m_2026_09, mp.m_2026_10, mp.m_2026_11, mp.m_2026_12, mp.m_2027_01
    , b.days_to_go_live, b.days_to_onboarding
    , b.warning_rules, b.has_warnings, b.hygiene_rules, b.has_hygiene_issues

    from gtm_silver.use_case_detail as c
    inner join usecases_filtered as b
    on c.usecase_id = b.usecase_id
    inner join quartely_projection_pivoted as qp
    on qp.usecase_id = c.usecase_id
    inner join monthly_projection_pivoted as mp
    on mp.usecase_id = c.usecase_id
)

select * from uco_view



